In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.functions import broadcast
import time

spark = SparkSession.builder \
    .master("local") \
    .appName("spark_homework") \
    .getOrCreate()

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

data_path = "../../data"

25/08/09 15:36:31 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
match_details = spark.read.option("header", "true").option("inferSchema", "true") \
    .csv(f"{data_path}/match_details.csv")

matches = spark.read.option("header", "true").option("inferSchema", "true") \
    .csv(f"{data_path}/matches.csv")

medals_matches_players = spark.read.option("header", "true").option("inferSchema", "true") \
    .csv(f"{data_path}/medals_matches_players.csv")

medals = spark.read.option("header", "true").option("inferSchema", "true") \
    .csv(f"{data_path}/medals.csv")

maps = spark.read.option("header", "true").option("inferSchema", "true") \
    .csv(f"{data_path}/maps.csv")
print("Data loaded")

Data loaded


In [3]:
match_details.createOrReplaceTempView("match_details_temp")
matches.createOrReplaceTempView("matches_temp")
medals_matches_players.createOrReplaceTempView("medals_matches_players_temp")

spark.sql("""
    CREATE OR REPLACE TABLE local.homework.match_details_bucketed
    USING iceberg
    PARTITIONED BY (bucket(16, match_id))
    AS SELECT * FROM match_details_temp
""")

spark.sql("""
    CREATE OR REPLACE TABLE local.homework.matches_bucketed
    USING iceberg
    PARTITIONED BY (bucket(16, match_id))
    AS SELECT * FROM matches_temp
""")

spark.sql("""
    CREATE OR REPLACE TABLE local.homework.medals_matches_players_bucketed
    USING iceberg
    PARTITIONED BY (bucket(16, match_id))
    AS SELECT * FROM medals_matches_players_temp
""")

print("Created tables")

25/08/09 15:36:43 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 16:===================>                                      (2 + 4) / 6]

Created tables


In [34]:
match_details_bucketed = spark.table("local.homework.match_details_bucketed")
matches_bucketed = spark.table("local.homework.matches_bucketed")
medals_matches_players_bucketed = spark.table("local.homework.medals_matches_players_bucketed")

match_with_details = match_details_bucketed.join(
    matches_bucketed, 
    "match_id", 
    "inner"
)

full_match_data = match_with_details.join(
    medals_matches_players_bucketed,
    ["match_id", "player_gamertag"],
    "left"
)

final_data = full_match_data \
    .join(broadcast(medals.select("medal_id", col("name").alias("medal_name"), \
                                  col("description").alias("medal_description"))), "medal_id", "left") \
    .join(broadcast(maps.select("mapid", col("name").alias("map_name"), \
                                col("description").alias("map_description"))), "mapid", "left")

final_data.show(3)

print(f"Final data has {final_data.count()} rows")

+--------------------+----------+--------------------+---------------+---------------------+------------+-----------------+--------+-----------------+------------------------+------------+---------------------------------+-----------------+----------------+-----------------------+-----------+--------------------------------+----------------+-------------------+---------------+-------------------+------------------+----------------------+--------------------------+-------------------------+------------------------+-------------------------+---------------------------+-------------------------------+--------------------------------+---------------------------+--------------------------------+-------------------------------+-------------------+--------------------+--------------------------+-------+-------+------------+--------------------+--------------------+-------------+-------------------+--------------+---------+--------------------+-----+-------------+--------------------+--------+

In [35]:
kills_per_player = final_data.groupBy("player_gamertag") \
    .agg(
        avg("player_total_kills").alias("avg_kills_per_game"),
        count("match_id").alias("games_played")
    ) \
    .filter(col("games_played") >= 5) \
    .orderBy(desc("avg_kills_per_game"))
print("Which player averages the most kills per game?")
kills_per_player.show(10)

Which player averages the most kills per game?
+---------------+------------------+------------+
|player_gamertag|avg_kills_per_game|games_played|
+---------------+------------------+------------+
|   gimpinator14|             109.0|          12|
|  I Johann117 I|              96.0|          15|
|BudgetLegendary|              83.0|          14|
|   TameablePoet|              82.5|          16|
|      GsFurreal|              75.0|          15|
|   Sexy is Back|              73.0|          15|
|     Profit TKO| 70.92857142857143|          14|
|   killerguy789|              68.0|          16|
|THC GUILTYSPARK|              67.0|          16|
|  DBossCnDTEXAS|              66.2|          15|
+---------------+------------------+------------+
only showing top 10 rows



In [36]:
playlist_popularity = final_data.groupBy("playlist_id") \
    .agg(
        countDistinct("match_id").alias("total_matches"),
        countDistinct("player_gamertag").alias("unique_players")
    ) \
    .orderBy(desc("total_matches"))
print("Which playlist gets played the most?")
playlist_popularity.show(10)

Which playlist gets played the most?
+--------------------+-------------+--------------+
|         playlist_id|total_matches|unique_players|
+--------------------+-------------+--------------+
|f72e0ef0-7c4a-430...|         7657|         22057|
|2323b76a-db98-4e0...|         3174|          9919|
|892189e9-d712-4bd...|         1974|          9779|
|c98949ae-60a8-43d...|         1828|          8887|
|f27a65eb-2d11-496...|          682|          3476|
|d0766624-dbd7-453...|          619|          2537|
|0bcf2be1-3168-4e4...|          562|          6868|
|5728f612-3f20-445...|          496|          1031|
|780cc101-005c-4fc...|          489|          6101|
|355dc154-9809-4ed...|          271|          1605|
+--------------------+-------------+--------------+
only showing top 10 rows



In [37]:
map_popularity = final_data.groupBy("mapid", "map_name") \
    .agg(
        countDistinct("match_id").alias("total_matches"),
        countDistinct("player_gamertag").alias("unique_players")
    ) \
    .orderBy(desc("total_matches"))
print("Which map gets played the most?")
map_popularity.show(10)

Which map gets played the most?


[Stage 726:============================>                            (3 + 3) / 6]

+--------------------+--------------+-------------+--------------+
|               mapid|      map_name|total_matches|unique_players|
+--------------------+--------------+-------------+--------------+
|c7edbf0f-f206-11e...|Breakout Arena|         7049|         21356|
|c74c9d0f-f206-11e...|        Alpine|         1387|         12403|
|cdb934b0-f206-11e...|        Empire|         1351|          6996|
|cb914b9e-f206-11e...|       The Rig|         1028|          5550|
|ce1dc2de-f206-11e...|         Truth|          979|          5209|
|caacb800-f206-11e...|         Plaza|          952|          5293|
|c7805740-f206-11e...|       Glacier|          933|          7949|
|cdee4e70-f206-11e...|        Regret|          927|          4966|
|cebd854f-f206-11e...|      Coliseum|          913|          5331|
|cd844200-f206-11e...|          Eden|          875|          4780|
+--------------------+--------------+-------------+--------------+
only showing top 10 rows



In [38]:
killing_spree_maps = final_data \
    .filter(col("medal_name").like("%Killing Spree%")) \
    .groupBy("mapid", "map_name", "medal_name") \
    .agg(
        count("medal_id").alias("killing_spree_count"),
        countDistinct("player_gamertag").alias("unique_players")
    ) \
    .orderBy(desc("killing_spree_count"))
print("Which map do players get the most Killing Spree medals on?")
killing_spree_maps.show(10)

Which map do players get the most Killing Spree medals on?
+--------------------+--------------+-------------+-------------------+--------------+
|               mapid|      map_name|   medal_name|killing_spree_count|unique_players|
+--------------------+--------------+-------------+-------------------+--------------+
|c7edbf0f-f206-11e...|Breakout Arena|Killing Spree|               6553|          3643|
|c74c9d0f-f206-11e...|        Alpine|Killing Spree|               4317|          3553|
|c7805740-f206-11e...|       Glacier|Killing Spree|               2611|          2132|
|cdb934b0-f206-11e...|        Empire|Killing Spree|               1991|          1544|
|ce1dc2de-f206-11e...|         Truth|Killing Spree|               1751|          1360|
|cb914b9e-f206-11e...|       The Rig|Killing Spree|               1733|          1358|
|caacb800-f206-11e...|         Plaza|Killing Spree|               1654|          1266|
|cebd854f-f206-11e...|      Coliseum|Killing Spree|               1646|

In [39]:

sorted_by_playlist = final_data.sortWithinPartitions("playlist_id")
sorted_by_map = final_data.sortWithinPartitions("mapid")
sorted_by_match = final_data.sortWithinPartitions("match_id")
sorted_by_combined = final_data.sortWithinPartitions("playlist_id", "mapid")

final_data.write.mode("overwrite").parquet("../../warehouse/unsorted")
sorted_by_playlist.write.mode("overwrite").parquet("../../warehouse/sorted_by_playlist")
sorted_by_map.write.mode("overwrite").parquet("../../warehouse/sorted_by_map")
sorted_by_match.write.mode("overwrite").parquet("../../warehouse/sorted_by_match")
sorted_by_combined.write.mode("overwrite").parquet("../../warehouse/sorted_by_combined")


final_data.write.mode("overwrite").saveAsTable("local.homework.unsorted")
sorted_by_playlist.write.mode("overwrite").saveAsTable("local.homework.sorted_by_playlist")
sorted_by_map.write.mode("overwrite").saveAsTable("local.homework.sorted_by_map")
sorted_by_match.write.mode("overwrite").saveAsTable("local.homework.sorted_by_match")
sorted_by_combined.write.mode("overwrite").saveAsTable("local.homework.sorted_by_combined")

print("Different sorted data written again")

Different sorted data written again


In [40]:
%%sql

SELECT SUM(file_size_in_bytes) as size, COUNT(1) as num_files, 'sorted_by_playlist' 
FROM local.homework.sorted_by_playlist.files
UNION ALL
SELECT SUM(file_size_in_bytes) as size, COUNT(1) as num_files, 'sorted_by_map' 
FROM local.homework.sorted_by_map.files
UNION ALL
SELECT SUM(file_size_in_bytes) as size, COUNT(1) as num_files, 'sorted_by_match' 
FROM local.homework.sorted_by_match.files
UNION ALL
SELECT SUM(file_size_in_bytes) as size, COUNT(1) as num_files, 'sorted_by_combined' 
FROM local.homework.sorted_by_combined.files
UNION ALL
SELECT SUM(file_size_in_bytes) as size, COUNT(1) as num_files, 'unsorted' 
FROM local.homework.unsorted.files

size,num_files,sorted_by_playlist
20744417,6,sorted_by_playlist
21217082,6,sorted_by_map
20496339,6,sorted_by_match
20456236,6,sorted_by_combined
20496339,6,unsorted
